In [1]:
import cupy as cp
import numpy as np
import os
import sys
sys.path.append(r"C:\Users\Dinesh\UGP\bayesian_nmor")
from Alternating_Estimation.core import *
import time
from joint_posterior import JointPosterior
from joint_estimator import JointEstimator
from joint_kl import choose_next_bias

In [2]:
DATA_DIR = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10"
#By scan 
scan_index = 0
bz_prior_range = 2
sigma_noise  = 0.4
#start time, curr_time
curr_time = 0.0
DATASET3 = DataContext(
                            sim_time = os.path.join(DATA_DIR, "t_array.csv"),
                            sim_freq = os.path.join(DATA_DIR, "delz_MHz.csv"),
                        sim_by = os.path.join(DATA_DIR, "dely_MHz.csv"),
                        sim_intensities = sorted([
                            os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.startswith("transH_t_vs_Bz_alpha2500_delx_0_dely_") and f.endswith(".csv")
                        ], key=lambda x: float(x.split("_dely_")[1].split(".csv")[0])),
                        exp_time = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\t_array_ Copy.csv",
                        exp_freq_axis = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_Y_MHz.csv",
                        exp_data = f"C:\\Users\\Dinesh\\UGP\\bayesian_nmor\\DataFiles_to_Dinesh_Pranav\\Data_files\\Experiment\\26_03_2026_CW_data\\Bz_V_Readout_By_{scan_index}.csv",
                        interpolator=r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10\gpu_sim_interpolator_new_normalisation_linear_spline_0.3_threshold.npz",
                        Sim_Aligned = True,
                        Exp_Aligned = False,
                        sim_pulse_thresh=0.3, 
                        exp_pulse_thresh=0.3
                        )
DEFAULT_ALTOPT_PARAMS = ParameterContext(
                            num_iter=10,
                            curr_time = curr_time,
                            max_time = 70.0,
                            tol_bz=1e-20,
                            tol_by=1e-20,
                            t_step = 0.2,   
                            fixed_by_estimate=0.0,
                            fixed_bz_estimate=0.0,
                            B_unk_bound_transverse_upper=0.5,
                            B_unk_bound_longitudinal = bz_prior_range,
                            print_plot = False,
                            likelihood_mode_longitudinal="Gaussian",
                            likelihood_mode_transverse="Gaussian",
                            sigma_noise_longitudinal = sigma_noise,
                            sigma_noise_transverse = sigma_noise
                            )

In [3]:
t_sim, f_sim, by_sim, full_interp = get_final_interpolator(DATASET3, DEFAULT_ALTOPT_PARAMS)
t_exp, f_bias_axis, exp_matrix = load_experiment(DATASET3, DATASET3.Exp_Aligned)
t_exp, f_bias_axis, exp_matrix = cp.asarray(t_exp), cp.asarray(f_bias_axis), cp.asarray(exp_matrix)


# bz_support = cp.arange(-DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.init_resolution_longitudinal)  # type: ignore
# curr_res_z = DEFAULT_ALTOPT_PARAMS.init_resolution_longitudinal
# by_support = cp.arange(DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_lower, DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_upper, DEFAULT_ALTOPT_PARAMS.init_resolution_transverse)  # type: ignore
# curr_res_y = DEFAULT_ALTOPT_PARAMS.init_resolution_transverse
bz_support = cp.linspace(-DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, 100)  # setting to 100 for simplicity

by_support = cp.linspace(DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_lower, DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_upper, 100)  # setting to 100 for simplicity


curr_bias = 0.0
start_time = DEFAULT_ALTOPT_PARAMS.curr_time

...Building Final Interpolator...
Time taken to load Interpolator: 3.0573713779449463
Loading Experiment from C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_V_Readout_By_0.csv...
  > Exp Start: -0.1100s
0.19826149940490723 was the time taken to load exp data.


In [4]:
posterior = JointPosterior(
    bz_support,
    by_support,
)

estimator = JointEstimator(
    posterior,
    full_interp,
    DEFAULT_ALTOPT_PARAMS,
)

Joint Bayesian Update Loop - We'll turn this into a nice function later once this executes.

In [5]:

history = {
    "time": [],
    "map_bz": [],
    "map_by": [],
    "std_bz": [],
    "std_by": [],
    "bias": [],
    "expectedkl": [],
    "posteriors": [],
}

time_cursor = DEFAULT_ALTOPT_PARAMS.curr_time
curr_bias = 0.0

start_wall = time.time()
loop_index = 0

while time_cursor < DEFAULT_ALTOPT_PARAMS.max_time:

    # ---------------------------------------------------------
    # Current control interval
    # ---------------------------------------------------------
    print(loop_index)
    loop_index += 1

    t_next = time_cursor + DEFAULT_ALTOPT_PARAMS.t_step

    idx_start = cp.searchsorted(
        t_exp,
        cp.asarray(time_cursor),
    )

    idx_end = cp.searchsorted(
        t_exp,
        cp.asarray(t_next),
    )

    if idx_start >= idx_end:
        break

    # ---------------------------------------------------------
    # Snap bias to nearest available experiment column
    # ---------------------------------------------------------

    bias_idx = cp.abs(
        f_bias_axis - curr_bias
    ).argmin()

    curr_bias = f_bias_axis[bias_idx]

    # ---------------------------------------------------------
    # Experimental measurement
    # ---------------------------------------------------------

    y_obs = exp_matrix[
        idx_start:idx_end,
        bias_idx,
    ]

    t_chunk = t_exp[
        idx_start:idx_end
    ]

    # ---------------------------------------------------------
    # Bayesian update
    # ---------------------------------------------------------

    summary = estimator.update(
        measurement=y_obs,
        times=t_chunk,
        bias=curr_bias,
    )

    # ---------------------------------------------------------
    # Save history
    # ---------------------------------------------------------

    history["time"].append(float(t_next))

    history["map_bz"].append(
        float(summary.map_bz)
    )

    history["map_by"].append(
        float(summary.map_by)
    )

    history["std_bz"].append(
        float(summary.std_bz)
    )

    history["std_by"].append(
        float(summary.std_by)
    )

    history["bias"].append(
        float(curr_bias)
    )

    history["posteriors"].append(
        estimator.posterior.weights.get()
    )

    print(
        f"T={t_next:.2f} "
        f"| Bias={curr_bias:.3f} "
        f"| MAP=({summary.map_bz:.4f}, "
        f"{summary.map_by:.4f}) "
        f"| Std=({summary.std_bz:.4f}, "
        f"{summary.std_by:.4f})"
    )

    # ---------------------------------------------------------
    # Candidate bias range
    # ---------------------------------------------------------

    f_index_1 = cp.where(
        f_bias_axis > f_sim[10]
    )[0][0]

    f_index_2 = cp.where(
        f_bias_axis > f_sim[-10]
    )[0][0]

    candidate_biases = f_bias_axis[
        f_index_1:f_index_2
    ]

    # ---------------------------------------------------------
    # Adaptive design
    # ---------------------------------------------------------

    curr_bias, expectedkl = choose_next_bias(
        posterior=estimator.posterior,
        candidate_biases=candidate_biases,
        current_time=t_next,
        next_time=t_next + DEFAULT_ALTOPT_PARAMS.t_step,
        interpolator=full_interp,
        params=DEFAULT_ALTOPT_PARAMS,
    )

    history["expectedkl"].append(
        expectedkl.get()
    )

    # ---------------------------------------------------------
    # Advance replay
    # ---------------------------------------------------------

    time_cursor = t_next

print(
    f"\nFinished in "
    f"{time.time()-start_wall:.2f} s"
)

0
T=0.20 | Bias=0.000 | MAP=(-1.5152, 0.2475) | Std=(1.1837, 0.1377)
1
T=0.40 | Bias=1.830 | MAP=(-1.5152, 0.3232) | Std=(1.1621, 0.1318)
2
T=0.60 | Bias=1.607 | MAP=(-1.5152, 0.3434) | Std=(1.1541, 0.1266)
3
T=0.80 | Bias=1.160 | MAP=(-1.4747, 0.3333) | Std=(1.1547, 0.1226)
4
T=1.00 | Bias=-1.250 | MAP=(-1.5960, 0.3939) | Std=(1.1360, 0.1180)
5
T=1.20 | Bias=-1.696 | MAP=(-1.0303, 0.2778) | Std=(1.1073, 0.1199)
6
T=1.40 | Bias=-1.383 | MAP=(-1.0303, 0.2727) | Std=(1.0935, 0.1157)
7
T=1.60 | Bias=1.294 | MAP=(-1.0303, 0.2727) | Std=(1.0840, 0.1088)
8
T=1.80 | Bias=-0.268 | MAP=(-1.0303, 0.2626) | Std=(1.0875, 0.1085)
9
T=2.00 | Bias=0.000 | MAP=(-0.7071, 0.2576) | Std=(1.0760, 0.1158)
10
T=2.20 | Bias=-0.045 | MAP=(-1.0303, 0.2576) | Std=(1.0454, 0.1045)
11
T=2.40 | Bias=-0.089 | MAP=(-1.0303, 0.2576) | Std=(1.0246, 0.0850)
12
T=2.60 | Bias=0.134 | MAP=(-1.1111, 0.2576) | Std=(0.8816, 0.0848)
13
T=2.80 | Bias=-0.179 | MAP=(-1.1111, 0.2576) | Std=(0.7461, 0.0891)
14
T=3.00 | Bias=0.000 

KeyboardInterrupt: 